# 05 — Evaluation (MLflow tracing + dual-LLM comparison)

**Owner:** Marston Ward (AIE / Team Lead) · **Course:** AAI-510 · Group 8

This notebook is the **evaluation evidence** for the agent. It uses **MLflow**
(a recognized tracing/eval platform) to capture traces and metrics, then runs a
**same-trace, two-LLM comparison**.

## What it produces (maps to the graded rubric)
1. **5 trace examples** — 3 in-scope attack events + **2 out-of-scope rejections**,
   each logged as an MLflow run/trace with params, metrics, and tags.
2. **Same-trace 2-LLM comparison** — the identical inputs are sent through **two
   Databricks-served models** read from config (`LLM_MODEL` vs `LLM_MODEL_B`),
   with a side-by-side results table and written analysis.

## Configurable models (no code edits)
Both model names come from config, so a grader can point the comparison at **any
two endpoints** via env: `LLM_MODEL` and `LLM_MODEL_B` (and `LLM_PROVIDER`,
`LLM_BASE_URL`, `LLM_API_KEY`/`DATABRICKS_TOKEN`).

## Mock-mode note (zero creds)
With no creds, the two "models" are **deterministic, calibrated stand-ins** so the
comparison mechanics run end-to-end offline. The numbers below are therefore
*simulated*; set the env vars above to get **live** numbers from two real
Databricks Model Serving endpoints. Helpers live in
`src/soc_agent/eval_helpers.py`.


In [1]:
# --- bootstrap: make src/soc_agent importable + default to MOCK_MODE ---
import os, sys
from pathlib import Path

# Walk up to the repo root (the dir that contains src/soc_agent).
_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "src" / "soc_agent").exists():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate src/soc_agent from " + str(_here))

sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)  # so ./mlruns and relative paths are consistent

# Default to zero-creds mock mode unless the grader set real env vars.
os.environ.setdefault("SOC_MOCK_MODE", "1")
os.environ.setdefault("MLFLOW_TRACKING_URI", "file:./mlruns")

from soc_agent import config
SETTINGS = config.get_settings()
print("Repo root        :", REPO_ROOT)
print("MOCK_MODE        :", SETTINGS.mock_mode)
print("LLM provider     :", SETTINGS.llm_provider, "(effective:", SETTINGS.effective_llm_provider() + ")")
print("Live Databricks  :", SETTINGS.use_live_databricks())


Repo root        : /Users/marston.ward/Documents/GitHub/msaai-510-group8-soc-triage-agent
MOCK_MODE        : True
LLM provider     : databricks (effective: mock)
Live Databricks  : False


In [2]:
from soc_agent import eval_helpers, gold_tools, config
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

mlf = eval_helpers.setup_mlflow(SETTINGS)
print("MLflow tracking URI:", SETTINGS.mlflow_tracking_uri)
print("Experiment         :", SETTINGS.mlflow_experiment)
print("Model A (LLM_MODEL )  :", SETTINGS.llm_model)
print("Model B (LLM_MODEL_B) :", SETTINGS.llm_model_b)


/Users/marston.ward/Documents/GitHub/msaai-510-group8-soc-triage-agent/_working/.pixi/envs/default/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/marston.ward/Documents/GitHub/msaai-510-group8-soc-triage-agent/_working/.pixi/envs/default/lib/python3.14/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/06/02 12:34:48 INFO mlflow.tracking.fluent: Experiment with name 'soc_triage_agent' does not exist. Creating a new experiment.


MLflow tracking URI: file:./mlruns
Experiment         : soc_triage_agent
Model A (LLM_MODEL )  : databricks-meta-llama-3-1-70b-instruct
Model B (LLM_MODEL_B) : databricks-dbrx-instruct


## 1) Five traces

Each scenario runs the full agent and is captured as an MLflow trace/run.

In [3]:
gold_tools.reset_incident_store()
traces = eval_helpers.run_traces(settings=SETTINGS)
traces


,scenario,expect,decision,tactic,technique_id,confidence,priority,severity,z_score,iterations,latency_ms,incident_written
0,credential_access_ws5,escalate,escalated,Credential Access,T1110,0.85,3.0,MEDIUM,3.82,4,531.2,True
1,execution_powershell_ws6,escalate,escalated,Execution,T1059,0.82,5.0,CRITICAL,4.51,4,329.4,True
2,persistence_service_filesrv1,escalate,escalated,Persistence,T1543,0.88,4.0,HIGH,3.10,4,311.4,True
3,out_of_scope_weather,reject,rejected,None,None,NaN,NaN,None,NaN,0,1.5,False
4,out_of_scope_recipe,reject,rejected,None,None,NaN,NaN,None,NaN,0,1.4,False


### Trace commentary

- **In-scope attack events** (`credential_access_ws5`, `execution_powershell_ws6`,
  `persistence_service_filesrv1`) are correctly **escalated**: the agent runs the
  full ReAct loop (4 tool iterations), the z-score exceeds 2.5, classifier
  confidence exceeds 0.7, and an incident is written.
- **Out-of-scope queries** (`out_of_scope_weather`, `out_of_scope_recipe`) are
  **rejected at the scope guard in 0 iterations** — no tools called, no LLM cost,
  no incident written. This is the graceful out-of-scope handling the rubric asks
  for.
- **Latency** is dominated by tool I/O (NVD calls when online); mock tools make
  runs fast and deterministic.

In [4]:
# Quick sanity assertions on the trace table.
esc = traces[traces["expect"] == "escalate"]
rej = traces[traces["expect"] == "reject"]
print("Escalated as expected :", (esc["decision"] == "escalated").all())
print("Rejected as expected  :", (rej["decision"] == "rejected").all())
print("Incidents written     :", len(gold_tools.get_incident_store()))


Escalated as expected : True
Rejected as expected  : True
Incidents written     : 3


## 2) Same-trace, two-LLM comparison

The identical scenario inputs are sent through **Model A** (`LLM_MODEL`) and
**Model B** (`LLM_MODEL_B`). We compare tactic, confidence, priority, and latency.

In [5]:
dual = eval_helpers.dual_llm_table(settings=SETTINGS)
dual


,scenario,slot,model,provider,decision,tactic,technique_id,confidence,priority,severity,latency_ms
0,credential_access_ws5,model_a,databricks-meta-llama-3-1-70b-instruct,mock,escalated,Credential Access,T1110,0.85,3,MEDIUM,221.3
1,credential_access_ws5,model_b,databricks-dbrx-instruct,mock,escalated,Credential Access,T1110,0.78,3,MEDIUM,223.8
2,execution_powershell_ws6,model_a,databricks-meta-llama-3-1-70b-instruct,mock,escalated,Execution,T1059,0.82,5,CRITICAL,974.2
3,execution_powershell_ws6,model_b,databricks-dbrx-instruct,mock,escalated,Execution,T1059,0.75,5,CRITICAL,992.0
4,persistence_service_filesrv1,model_a,databricks-meta-llama-3-1-70b-instruct,mock,escalated,Persistence,T1543,0.88,4,HIGH,989.6
5,persistence_service_filesrv1,model_b,databricks-dbrx-instruct,mock,escalated,Persistence,T1543,0.81,4,HIGH,989.6


In [6]:
summary = eval_helpers.comparison_summary(dual)
summary


,model,mean_confidence,mean_priority,mean_latency_ms,n,tactic_agreement_rate
0,databricks-dbrx-instruct,0.78,4.0,735.133333,3,1.0
1,databricks-meta-llama-3-1-70b-instruct,0.85,4.0,728.366667,3,1.0


### Comparison analysis (performance observations + outcomes)

*(In mock mode these are simulated calibrations; with live endpoints they reflect
real model behavior.)*

- **Tactic agreement.** Both models agree on the MITRE tactic for every scenario
  (`tactic_agreement_rate = 1.0`). The tactic is anchored by the deterministic
  rule signal from Sai's `classify_threat`, so model choice mainly affects
  **confidence calibration** and **prose**, not the core label.
- **Confidence calibration.** Model A (Llama-3.1-70B) reports **higher mean
  confidence** than Model B (DBRX) on identical inputs — useful when tuning the
  `min_confidence_to_ticket` escalation threshold per model.
- **Latency.** Mean latency differs between endpoints; in production this trades
  off against cold-start behavior for the larger model (a documented risk in the
  proposal — keep the endpoint warm).
- **Human-in-the-loop.** Tickets below threshold or that fail JSON validation are
  routed to **manual review**, so a human analyst remains the final arbiter — the
  evaluation measures how much triage the agent can safely automate, not full
  autonomy.

### How a grader gets live numbers
```bash
export SOC_MOCK_MODE=0 LLM_PROVIDER=databricks
export LLM_BASE_URL="https://<workspace>.cloud.databricks.com/serving-endpoints"
export DATABRICKS_TOKEN="dapi..."
export LLM_MODEL="databricks-meta-llama-3-1-70b-instruct"
export LLM_MODEL_B="databricks-dbrx-instruct"
```
Then re-run this notebook — the same code compares the two live endpoints.

In [7]:
# Persist the comparison artifacts for the report.
import os
os.makedirs("docs/eval_artifacts", exist_ok=True)
traces.to_csv("docs/eval_artifacts/traces.csv", index=False)
dual.to_csv("docs/eval_artifacts/dual_llm_comparison.csv", index=False)
summary.to_csv("docs/eval_artifacts/dual_llm_summary.csv", index=False)
print("Saved trace + comparison CSVs under docs/eval_artifacts/.")
print("MLflow runs under:", SETTINGS.mlflow_tracking_uri)


Saved trace + comparison CSVs under docs/eval_artifacts/.
MLflow runs under: file:./mlruns
